# Qdrant VectorDB: Semantic Search for Pins

Research goal: set up Qdrant with named vectors (text + image) per pin and evaluate ANN search quality.


In [ ]:
# Requires Qdrant running locally or via Docker:
# docker run -d -p 6333:6333 -p 6334:6334 qdrant/qdrant:v1.9.1
# !pip install qdrant-client==1.9.1 sentence-transformers


In [ ]:
import numpy as np
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    Filter, FieldCondition, MatchValue
)
from sentence_transformers import SentenceTransformer

qdrant = QdrantClient(url="http://localhost:6333")
embed_model = SentenceTransformer("intfloat/multilingual-e5-large")
print("Qdrant client and embedding model ready.")


## 1. Create collection with named vectors

In [ ]:
COLLECTION = "pins_research"

if qdrant.collection_exists(COLLECTION):
    qdrant.delete_collection(COLLECTION)

qdrant.create_collection(
    collection_name=COLLECTION,
    vectors_config={
        # E5-large text embeddings
        "text":  VectorParams(size=1024, distance=Distance.COSINE),
        # CLIP image embeddings (placeholder dim=768, using random for demo)
        "image": VectorParams(size=768,  distance=Distance.COSINE),
    }
)
print(f"Collection {COLLECTION!r} created with named vectors: text (1024) + image (768)")


## 2. Upsert pins with text + image vectors

In [ ]:
pin_data = [
    {"id": 1, "pin_id": "p001", "title": "Закат на Байкале", "description": "Невероятные оттенки оранжевого на воде", "tags": ["природа", "закат"]},
    {"id": 2, "pin_id": "p002", "title": "Уличная еда в Стамбуле", "description": "Симит, балык экмек и турецкий чай", "tags": ["еда", "турция"]},
    {"id": 3, "pin_id": "p003", "title": "Горы Алтая", "description": "Катунь в сентябре — бирюзовая вода", "tags": ["горы", "поход"]},
    {"id": 4, "pin_id": "p004", "title": "Ночная Москва", "description": "Арт-объекты и неоновые надписи", "tags": ["ночь", "арт"]},
    {"id": 5, "pin_id": "p005", "title": "Santorini Sunset", "description": "White-washed cliffs at golden hour", "tags": ["sunset", "greece"]},
    {"id": 6, "pin_id": "p006", "title": "Tokyo Night Photography", "description": "Neon lights and rain in Shinjuku", "tags": ["night", "japan"]},
    {"id": 7, "pin_id": "p007", "title": "Dolomites Hiking", "description": "Tre Cime loop on August morning", "tags": ["hiking", "italy"]},
    {"id": 8, "pin_id": "p008", "title": "Melbourne Coffee", "description": "Specialty coffee in laneway cafes", "tags": ["coffee", "australia"]},
]

texts = [f"passage: {p["title"]} {p["description"]}" for p in pin_data]
text_vecs = embed_model.encode(texts, normalize_embeddings=True).tolist()

# Simulate image vectors (replace with real CLIP embeddings in production)
np.random.seed(42)
image_vecs = (np.random.randn(len(pin_data), 768)).tolist()
image_vecs = [v / np.linalg.norm(v) for v in image_vecs]

points = [
    PointStruct(
        id=p["id"],
        vector={"text": text_vecs[i], "image": image_vecs[i]},
        payload={k: v for k, v in p.items() if k != "id"}
    )
    for i, p in enumerate(pin_data)
]

qdrant.upsert(collection_name=COLLECTION, points=points)
print(f"Upserted {len(points)} points.")


## 3. Semantic search with filtering

In [ ]:
def semantic_search(query, vector_name="text", limit=3, tag_filter=None):
    q_vec = embed_model.encode(f"query: {query}", normalize_embeddings=True).tolist()
    filt = None
    if tag_filter:
        filt = Filter(must=[FieldCondition(key="tags", match=MatchValue(value=tag_filter))])
    results = qdrant.search(
        collection_name=COLLECTION,
        query_vector=(vector_name, q_vec),
        limit=limit,
        query_filter=filt,
        with_payload=True,
    )
    return [(r.id, round(r.score, 3), r.payload["title"]) for r in results]

print("--- Semantic search examples ---")
for q in ["снимок заката над водой", "горный поход в России", "urban night photography", "coffee shop cozy"]:
    hits = semantic_search(q)
    print(f"Q: {q!r}")
    for h in hits:
        print(f"   {h}")
    print()

print("--- Filtered by tag='горы' ---")
print(semantic_search("природа поход", tag_filter="горы"))


## 4. Collection stats and index info

In [ ]:
info = qdrant.get_collection(COLLECTION)
print(f"Points count: {info.points_count}")
print(f"Vectors count: {info.vectors_count}")
print(f"Status: {info.status}")
print(f"Config: {info.config.params.vectors}")


## Conclusions

- Named vectors allow storing text + image embeddings per pin in one collection
- Tag/user_id payload filtering works at index level (no post-filter performance hit)
- ANN search returns relevant results for cross-lingual queries (RU query → EN pin)
- **Next:** combine ES BM25 + Qdrant ANN with Reciprocal Rank Fusion in 
